In [16]:
import numpy as np
import csv

# 1. Leitura do CSV


In [17]:
def load_data(filename):
    X = []
    y = []

    with open(filename, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)

        for row in reader:
            # Target
            survived = row["survived"]
            if survived == "":
                continue

            y.append(int(survived))

            # Features Selecionadas
            # Vamos usar:
            # pclass, sex, age, sibsp, parch, fare

            pclass = float(row["pclass"]) if row["pclass"] != "" else 0

            # Convertendo Sexo
            sex = 1 if ["sex"] == "female" else 0

            age = float(row["age"]) if row["age"] != "" else -1

            sibsp = float(row["sibsp"]) if row["sibsp"] != "" else 0
            parch = float(row["parch"]) if row["parch"] != "" else 0
            fare = float(row["fare"]) if row["fare"] != "" else 0

            X.append([pclass, sex, age, sibsp, parch, fare])

    return np.array(X), np.array(y).reshape(-1, 1)

# 2. Tratamento de Dados


In [18]:
def fill_missing_age(X):
    ages = X[:, 2]
    mean_age = np.mean(ages != -1)

    X[:2], np.where(ages == -1, mean_age, ages)
    return X

In [19]:
def normalize(X):
    mean = np.mean(X, axis=0)
    std = np.std(X, axis=0)
    return (X - mean) / (std + 1e-8)

In [20]:
def add_bias(X):
    ones = np.ones((X.shape[0], 1))
    return np.hstack((ones, X))

# 3. Funções do Modelo


In [21]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

In [22]:
def compute_cost(X, y, beta):
    m = len(y)
    h = sigmoid(X @ beta)
    epsilon = 1e-8

    cost = -(1 / m) * np.sum(
        y * np.log(h + epsilon) + (1 - y) * np.log(1 - h + epsilon)
    )
    
    return cost

In [23]:
def gradient_descent(X, y, beta, lr, epochs):
    m = len(y)

    for i in range(epochs):
        h = sigmoid(X @ beta)
        gradient = (1 / m) * (X.T @ (h - y))

        beta = beta - lr * gradient

        if i % 100 == 0:
            print(f"Epoch {i} - Cost: {compute_cost(X, y, beta):4f}")

    return beta

# 4. Treinamento


In [24]:
def train(X, y):
    X = fill_missing_age(X)
    X = normalize(X)
    X = add_bias(X)

    beta = np.zeros((X.shape[1], 1))

    beta = gradient_descent(X, y, beta, lr=0.01, epochs=2000)

    return beta, X

# 5. Predição


In [25]:
def predict(X, beta):
    probs = sigmoid(X @ beta)
    return (probs >= 0.5).astype(int)

In [26]:
def accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

# 6. Slip Treino / Teste


In [27]:
def train_test_split(X, y, test_size=0.2):
    np.random.seed(42)

    indices = np.arange(len(X))
    np.random.shuffle(indices)

    split = int(len(X) * (1 - test_size))

    train_idx = indices[:split]
    test_idx = indices[split:]

    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

# 7. Execução


In [ ]:
X, y = load_data("titanic.csv")

X_train, X_test, y_train, y_test = train_test_split(X, y)

beta, X_train_processed = train(X_train, y_train)

# Processar teste com mesmas transformações
X_test = fill_missing_age(X_test)
X_test = normalize(X_test)
X_test = add_bias(X_test)

y_pred = predict(X_test, beta)

acc = accuracy(y_test, y_pred)

print("\nAcurácia: ", acc)

Epoch 0 - Cost: 0.692568
Epoch 100 - Cost: 0.651347
Epoch 200 - Cost: 0.630033
Epoch 300 - Cost: 0.618005
Epoch 400 - Cost: 0.610714
Epoch 500 - Cost: 0.606037
Epoch 600 - Cost: 0.602899
Epoch 700 - Cost: 0.600718
Epoch 800 - Cost: 0.599156
Epoch 900 - Cost: 0.598012
Epoch 1000 - Cost: 0.597156
Epoch 1100 - Cost: 0.596505
Epoch 1200 - Cost: 0.596004
Epoch 1300 - Cost: 0.595612
Epoch 1400 - Cost: 0.595302
Epoch 1500 - Cost: 0.595056
Epoch 1600 - Cost: 0.594857
Epoch 1700 - Cost: 0.594697
Epoch 1800 - Cost: 0.594565
Epoch 1900 - Cost: 0.594458

Acurácia:  0.6564885496183206
